<a href="https://colab.research.google.com/github/Bhaveshmeghwal21/AutoLUT/blob/main/AutoLUT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/SuperKenVery/AutoLUT
%cd AutoLUT
!pip install pyyaml opencv-python

Cloning into 'AutoLUT'...
remote: Enumerating objects: 91, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 91 (delta 34), reused 88 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (91/91), 2.16 MiB | 24.06 MiB/s, done.
Resolving deltas: 100% (34/34), done.
/content/AutoLUT


In [ ]:
import os
os.makedirs("data", exist_ok=True)
%cd data
!rm -rf SRBenchmark

!wget https://cv.snu.ac.kr/research/EDSR/benchmark.tar
!tar -xvf benchmark.tar
!mv benchmark SRBenchmark

# Download DIV2K (High Resolution images)
!wget http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip
!unzip -q DIV2K_train_HR.zip

# Download DIV2K (Low Resolution images - 4x bicubic)
!wget http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_LR_bicubic_X4.zip
!unzip -q DIV2K_train_LR_bicubic_X4.zip

# Rename folders to match what the script expects
!mv DIV2K_train_HR DIV2K
!mv DIV2K_train_LR_bicubic/X4 DIV2K/LR
!rm -rf DIV2K_train_LR_bicubic

%cd ..
print("Datasets Downloaded Successfully!")

/content/AutoLUT/data
--2026-04-20 08:53:46--  https://cv.snu.ac.kr/research/EDSR/benchmark.tar
Resolving cv.snu.ac.kr (cv.snu.ac.kr)... 147.46.67.83
Connecting to cv.snu.ac.kr (cv.snu.ac.kr)|147.46.67.83|:443... connected.
Unable to establish SSL connection.
tar: benchmark.tar: Cannot open: No such file or directory
tar: Error is not recoverable: exiting now
mv: cannot stat 'benchmark': No such file or directory
--2026-04-20 08:53:47--  http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip
Resolving data.vision.ee.ethz.ch (data.vision.ee.ethz.ch)... 129.132.52.178, 2001:67c:10ec:36c2::178
Connecting to data.vision.ee.ethz.ch (data.vision.ee.ethz.ch)|129.132.52.178|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip [following]
--2026-04-20 08:53:47--  https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip
Connecting to data.vision.ee.ethz.ch (data.vision.ee.ethz.ch)|129.132.52.178|:443... c

In [ ]:
import os

# Navigate to the directory containing data.py
os.chdir('/content/AutoLUT/MuLUT+AutoLUT/sr/')

# Display the content of data.py
with open('data.py', 'r') as f:
    print(f.read())

import os
import random
import sys
import torch

import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader

sys.path.insert(0, "../")  # run under the project directory
from common.utils import modcrop
from loguru import logger



class DIV2K(Dataset):
    def __init__(self, scale, path, patch_size, rigid_aug=True, debug=False,gpuNum=1):
        super(DIV2K, self).__init__()
        self.scale = scale
        self.sz = patch_size
        self.rigid_aug = rigid_aug
        self.path = path
        self.file_list = [str(i).zfill(4)
                          for i in range(1, 901)]  # use both train and valid
        self.debug=debug
        self.gpuNum=gpuNum

        if not debug and gpuNum==1:
            # need about 8GB shared memory "-v '--shm-size 8gb'" for docker container
            self.hr_cache = os.path.join(path, "cache_hr.npy")
            if not os.path.exists(self.hr_cache):
                self.cache_hr()
                logger.info("HR im

In [ ]:
!pip install pyyaml opencv-python loguru jaxtyping typeguard==2.13.3 rich tensorboard pytorch-lightning tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 75.4 MB/s eta 0:00:00
  Attempting uninstall: typeguard
    Found existing installation: typeguard 4.5.1
    Uninstalling typeguard-4.5.1:
      Successfully uninstalled typeguard-4.5.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
inflect 7.5.0 requires typeguard>=4.0.1, but you have typeguard 2.13.3 which is incompatible.


In [ ]:
%cd /content/AutoLUT/data/

# 1. Delete any messy folders
!rm -rf DIV2K DIV2K_train_HR DIV2K_train_LR_bicubic

# 2. Explicitly create the exact folder structure needed
!mkdir -p DIV2K/HR
!mkdir -p DIV2K/LR/X4

# 3. Extract the images directly into those folders
!unzip -j -q DIV2K_train_HR.zip -d DIV2K/HR/
!unzip -j -q DIV2K_train_LR_bicubic_X4.zip -d DIV2K/LR/X4/



/content/AutoLUT/data


In [ ]:
import os
import shutil

hr_path = "/content/AutoLUT/data/DIV2K/HR"
lr_path = "/content/AutoLUT/data/DIV2K/LR/X4"

subset_hr = "/content/AutoLUT/data/DIV2K_subset/HR"
subset_lr = "/content/AutoLUT/data/DIV2K_subset/LR/X4"

os.makedirs(subset_hr, exist_ok=True)
os.makedirs(subset_lr, exist_ok=True)

# Only training images (0001–0800)
images = sorted([img for img in os.listdir(hr_path) if int(img[:4]) <= 800])

# Take first 1%
sample_size = max(1, int(0.01 * len(images)))
selected = images[:sample_size]

print(f"Using first {len(selected)} images")

# Copy WITHOUT renaming (safe now since sequential)
for img in selected:
    # HR
    shutil.copy(os.path.join(hr_path, img),
                os.path.join(subset_hr, img))

    # LR
    lr_img = img.replace(".png", "x4.png")
    shutil.copy(os.path.join(lr_path, lr_img),
                os.path.join(subset_lr, lr_img))

print("Subset created successfully!")

Using first 8 images
Subset created successfully!


In [ ]:
%cd /content/AutoLUT/MuLUT+AutoLUT/sr/

/content/AutoLUT/MuLUT+AutoLUT/sr


In [ ]:
%cd /content/AutoLUT/data/

# Download the missing Validation sets
!wget http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip
!wget http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_LR_bicubic_X4.zip

# Extract them directly into our existing folders so they join images 0001-0800
!unzip -j -q DIV2K_valid_HR.zip -d DIV2K/HR/
!unzip -j -q DIV2K_valid_LR_bicubic_X4.zip -d DIV2K/LR/X4/

print("Validation datasets added!")


/content/AutoLUT/data
--2026-04-06 12:26:17--  http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip
Resolving data.vision.ee.ethz.ch (data.vision.ee.ethz.ch)... 129.132.52.178, 2001:67c:10ec:36c2::178
Connecting to data.vision.ee.ethz.ch (data.vision.ee.ethz.ch)|129.132.52.178|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip [following]
--2026-04-06 12:26:18--  https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip
Connecting to data.vision.ee.ethz.ch (data.vision.ee.ethz.ch)|129.132.52.178|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 448993893 (428M) [application/zip]
Saving to: ‘DIV2K_valid_HR.zip’

DIV2K_valid_HR.zip  100%[===================>] 428.19M  20.5MB/s    in 22s     

2026-04-06 12:26:41 (19.2 MB/s) - ‘DIV2K_valid_HR.zip’ saved [448993893/448993893]

--2026-04-06 12:26:41--  http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_LR_bicubic_X4.zip
Re

In [ ]:
%cd /content/AutoLUT/MuLUT+AutoLUT/sr/

# Turn OFF the aggressive RAM caching
!sed -i 's/if not debug and gpuNum==1:/if False:/g' data.py

# Turn ON sequential Hard-Drive reading
!sed -i 's/if self.debug or self.gpuNum>1:/if True:/g' data.py

print("Safe Mode Enabled!")


/content/AutoLUT/MuLUT+AutoLUT/sr
Safe Mode Enabled!


In [ ]:
%cd /content/AutoLUT/MuLUT+AutoLUT/sr/

# 1. Tell data.py to ONLY validate on the DIV2K images
!sed -i "s/\['Set5', 'Set14', 'B100', 'Urban100', 'Manga109', 'DIV2K'\]/\['DIV2K'\]/g" data.py
!sed -i "s/assert (len(self.ims.keys())/pass #/g" data.py

# 2. Tell the main training script to do the same
!sed -i "s/\['Set5', 'Set14'\]/\['DIV2K'\]/g" 1_train_model.py
!sed -i "s/\['Set5', 'Set14', 'B100', 'Urban100', 'Manga109', 'DIV2K'\]/\['DIV2K'\]/g" 1_train_model.py

# 3. Re-route the internal saving parameter
!sed -i "s/dataset=='Set5'/dataset=='DIV2K'/g" 1_train_model.py

print("Done! Validation successfully pointed to DIV2K.")


/content/AutoLUT/MuLUT+AutoLUT/sr
Done! Validation successfully pointed to DIV2K.


In [ ]:
!mkdir -p /content/AutoLUT/data/SRBenchmark


In [ ]:
%cd /content/AutoLUT/MuLUT+AutoLUT/sr
!python 1_train_model.py \
    --stages 2 \
    --numSamplers 3 \
    --sampleSize 5 \
    --activateFunction gelu \
    -e ../models/autolut_mulut \
    --valDir ../../data/SRBenchmark \
    --trainDir ../../data/DIV2K \
    --no-lambda-lr

/content/AutoLUT/MuLUT+AutoLUT/sr
2026-04-06 12:26:58.895817: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775478419.095338    7019 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775478419.150335    7019 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775478419.550677    7019 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775478419.550734    7019 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775478419.550743    7019 computation_pla

In [ ]:
!python 2_transfer_to_lut.py \
    --loadIter 200000 \
    -e ../models/autolut_mulut

In [ ]:
!python 3_finetune_lut.py \
    --stages 2 \
    --numSamplers 3 \
    --sampleSize 5 \
    -e ../models/autolut_mulut/ \
    --no-lambda-lr \
    --valDir ../../data/SRBenchmark \
    --trainDir ../../data/DIV2K

In [ ]:
import shutil
from google.colab import files

# Zip the models folder which contains the LUT files
shutil.make_archive("/content/autolut_trained_models", 'zip', "/content/AutoLUT/MuLUT+AutoLUT/models")

# Download it
files.download("/content/autolut_trained_models.zip")